In [1]:
import poolparty as pp

rainbow_cycle = ['red', 'orange', 'yellow', 'lime', 'green', 'cyan', 'blue', 'indigo', 'darkviolet', 'magenta']
mut_style = 'lower white'
GB1_ORF = "ATGACCTACAAACTGATCCTGAACGGTAAAACCCTGAAAGGTGAAACCACCACCGAAGCTGTTGACGCTGCTACCGCAGAAAAAGTGTTCAAACAGTACGCTAACGACAACGGTGTTGACGGTGAATGGACCTACGACGACGCTACCAAAACCTTCACCGTTACCGAAAAACCGGAA"
GB1_ORF = GB1_ORF[:30]+GB1_ORF[-30:]

pp.init()

wt = pp.from_seq(GB1_ORF).\
    annotate_orf("gb1", frame=1, style_codons=rainbow_cycle).stylize(which='tags', style='gray')
wt.print_library()

pos = slice(1, 56)
single = wt.mutagenize_orf(
    num_mutations=1,
    codon_positions=pos, 
    mode='sequential',
    style=mut_style,
    prefix='single'
)
#single.print_library(num_seqs=10)

double = wt.mutagenize_orf(
    num_mutations=2,
    codon_positions=pos, 
    mode='sequential',
    style=mut_style,
    prefix='double'
)
#double.print_library(num_seqs=10)

random = wt.mutagenize_orf(
    mutation_rate=0.10,
    codon_positions=pos, 
    mode='random',
    style=mut_style,
    prefix='random',
    num_states=100_000)
#random.print_library(num_seqs=10)

dms_lib = pp.stack([
    wt.add_prefix('wt').repeat(times=100_000, prefix='v'),
    single.repeat(times=100, prefix='v'),
    double,
    random,
]).named('dms_lib')

display_slice = slice(42, None, 20_000)
dms_lib[display_slice].print_library()
print(dms_lib.num_states)


pool[3]: seq_length=60, num_states=1
seq
<gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>

pool[11]: seq_length=None, num_states=15
name             seq
wt.v_00042       <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
wt.v_20042       <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
wt.v_40042       <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
wt.v_60042       <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
wt.v_80042       <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
single_000.v_42  <gb1>ATGttcTACAAACTGATCCTGAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
single_200.v_42  <gb1>ATGACCTACAAACTGATCCTGAACGGTAAAAAAcagTTCACCGTTACCGAAAAACCGGAA</gb1>
double_03942     <gb1>ATGagaTACAAACTGATCCTGAACGGTAAAAAAACCcacACCGTTACCGAAAAACCGGAA</gb1>
double_23942     <gb1>ATGACCTACAAACTGaccatgAACGGTAAAAAAACCTTCACCGTTACCGAAAAACCGGAA</gb1>
double_43942     <gb1>A

In [2]:
protein_lib = dms_lib.translate(region='gb1', frame=1).named('protein_lib')
protein_lib[display_slice].print_library(chars_per_aa=1)
protein_lib[display_slice].print_library(chars_per_aa=3, aa_separator=' ')

pool[13]: seq_length=None, num_states=15
name             seq
wt.v_00042       MTYKLILNGKKTFTVTEKPE
wt.v_20042       MTYKLILNGKKTFTVTEKPE
wt.v_40042       MTYKLILNGKKTFTVTEKPE
wt.v_60042       MTYKLILNGKKTFTVTEKPE
wt.v_80042       MTYKLILNGKKTFTVTEKPE
single_000.v_42  MfYKLILNGKKTFTVTEKPE
single_200.v_42  MTYKLILNGKKqFTVTEKPE
double_03942     MrYKLILNGKKThTVTEKPE
double_23942     MTYKLtmNGKKTFTVTEKPE
double_43942     MTYKLILNGdKTFTVeEKPE
random_02211     MTYKLILcGKKTFgVTEKPE
random_22211     MTYKLILNGKKcFTgTEKPE
random_42211     MTYKrILNGKKnFTVvEgPE
random_62211     MTYKLILNGKKTFTVTEKPE
random_82211     MaYKLILNGKKTFTVTEKnE

pool[14]: seq_length=None, num_states=15
name             seq
wt.v_00042       Met Thr Tyr Lys Leu Ile Leu Asn Gly Lys Lys Thr Phe Thr Val Thr Glu Lys Pro Glu
wt.v_20042       Met Thr Tyr Lys Leu Ile Leu Asn Gly Lys Lys Thr Phe Thr Val Thr Glu Lys Pro Glu
wt.v_40042       Met Thr Tyr Lys Leu Ile Leu Asn Gly Lys Lys Thr Phe Thr Val Thr Glu Lys Pro Glu
wt.v_60042    

ProteinPool(id=14, name='pool[14]', op='op[14]:state_slice', num_states=15)

In [11]:
# Demonstrate Pool.to_file() for streaming export to different formats
import time
from pathlib import Path

# Create (and clear) output directory
outdir = Path("gb1_exports")
if outdir.exists():
    for f in outdir.iterdir():
        f.unlink()
outdir.mkdir(exist_ok=True)

kwargs = dict(num_seqs=10_000, seed=42) #dict(num_cycles=1, seed=42)

# dms_lib.to_df(**kwargs)
# dms_lib.to_file(outdir / "library.txt", **kwargs)
# dms_lib.to_file(outdir / "library.fasta", description=lambda row: f"len={len(row['seq'])}", **kwargs)
dms_lib.to_file(outdir / "library.csv.gz", include_design_cards=True, **kwargs)
#dms_lib.to_file(outdir / "library.jsonl", **kwargs)

# Show what was written
print(f"\nFiles created in {outdir}/:")
for f in sorted(outdir.iterdir()):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")


Exporting to library.csv.gz (format: csv.gz) ...


Generating sequences: 100%|██████████| 10000/10000 [00:02<00:00, 3461.76seq/s]


Files created in gb1_exports/:
  library.csv.gz: 27,737 bytes
